In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/shohanmahmudshimul/
      /kaggle/input/datasets/shohanmahmudshimul/sat-ext/
        /kaggle/input/datasets/shohanmahmudshimul/sat-ext/sat_ext/
          README.md
          audit.py
          config.py
          dataset.py
          infer.py
          metrics.py
          ... +7 more files
      /kaggle/input/datasets/shohanmahmudshimul/sat-splits/
        /kaggle/input/datasets/shohanmahmudshimul/sat-splits/audit/
        /kaggle/input/datasets/shohanmahmudshimul/sat-splits/splits/
      /kaggle/input/datasets/shohanmahmudshimul/satelite-coco/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/shohanmahmudshimul/satelite-coco/test/
          1011_jpeg.rf.141ad0f26ab02f1bfee50847febecb85.jpg
          1016_jpeg.rf.0aef8e3664a12b1681da06035362ed54.jpg
          1025_jpeg.rf.ce6b95f0170aab49e8aab8728fca22a8.jpg
          1067_jpeg.rf.b20f17d8c4416f8b8ed687b14a48966a.jpg
          1077_

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/shohanmahmudshimul/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
KEY = "B"            # preset - do not change
PILOT = False         # True = 1-epoch YOLO11n check of the whole GPU pipeline (tagged __pilot, ignored by the tables)
import plans, train
PLAN = plans.PILOT if PILOT else plans.PLANS[KEY]
EPOCHS = 1 if (PILOT or cfg.SMOKE) else None
TAG = "pilot" if PILOT else ""
if PILOT:
    print("PILOT: grouped / YOLO11n / seed 42, 1 epoch")
else:
    plans.show(KEY)
print("epochs:", EPOCHS or cfg.TRAIN_PARAMS["epochs"])

02B-train-rtdetr-l-grouped-s44-and-orig-s42: 2 runs, ~4.2 GPU h (planning estimate)
  split=grouped      model=RT-DETR-L  seed=44
  split=orig         model=RT-DETR-L  seed=42
epochs: 50


In [4]:
utils.require_gpu()
assert utils.install_detection_stack(), "pip install failed - is Internet ON? (Settings -> Internet)"
print(utils.versions())

GPU: {'device': 'cuda:0', 'gpu': 'Tesla T4', 'n_gpu': 2, 'torch': '2.11.0+cu128'}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{'ultralytics': '8.4.37', 'sahi': '0.12.8', 'torch': '2.11.0+cu128', 'pycocotools': 'installed', 'numpy': '2.1.3', 'cv2': '4.14.0'}


In [5]:
done = train.run_plan(PLAN, epochs=EPOCHS, tag=TAG)

materialized 'grouped' (f7cee44b1365) -> /kaggle/working/data_grouped  images {'train': 984, 'val': 80, 'test': 82}
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_grouped/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001429, lrf

In [6]:
import shutil, glob
for rid, status in done:
    ev = os.path.join(cfg.RESULTS_DIR, rid, "eval_test.json")
    if os.path.isfile(ev):
        d = utils.load_json(ev)
        u = d["ultralytics_test"]
        print(f"{rid:<30} test mAP50={u['mAP50']:.4f} mAP50-95={u['mAP50_95']:.4f} | " +
              " ".join(f"{m}:AP={v['coco'].get('AP_all', float('nan')):.3f}" for m, v in d["modes"].items()))
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, f"results_{KEY}"), "zip", cfg.RESULTS_DIR)
print("\nsmall results zip (no weights):", z, f"{os.path.getsize(z) / 1e6:.1f} MB")
for f in glob.glob(os.path.join(cfg.WORK_DIR, "runs")):
    shutil.rmtree(f, ignore_errors=True)       # Ultralytics run folders are duplicated in results/ + weights/

grouped__RT-DETR-L__s44        test mAP50=0.7316 mAP50-95=0.3489 | standard:AP=0.349 sahi480:AP=0.345 sahi320:AP=0.308 sahi160:AP=0.274
orig__RT-DETR-L__s42           test mAP50=0.6781 mAP50-95=0.3741 | standard:AP=0.376 sahi480:AP=0.355 sahi320:AP=0.318 sahi160:AP=0.253

small results zip (no weights): /kaggle/working/results_B.zip 57.9 MB
